# Clasificación de Texto con HuggingFace
### IPD434 — Computación Evolutiva e Inteligencia Artificial
**Universidad Técnica Federico Santa María**

---

[HuggingFace](https://huggingface.co/) es una plataforma que centraliza **modelos**, **datasets** y **aplicaciones** de código abierto de Machine Learning y Deep Learning. En este notebook usaremos su librería `transformers` para hacer **fine-tuning** de un modelo tipo BERT sobre la tarea de **análisis de sentimiento** del dataset IMDb.

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Comprender** qué ofrece la plataforma HuggingFace (modelos, datasets, tokenizers).
2. **Tokenizar** texto con `AutoTokenizer` y cargar modelos con `AutoModel`.
3. **Realizar** una tarea de clasificación de texto con `transformers`.
4. **Evaluar** el desempeño del modelo en la tarea de clasificación.

> 💡 **Prerrequisitos:** [04_RedesNeuronales](../../04_RedesNeuronales/04_RedesNeuronales.ipynb) y [Hugging Face Pipeline](Hugging%20Face%20Pipeline.ipynb).

## 1. El flujo de trabajo de *fine-tuning*

A diferencia del notebook de `pipeline` (donde solo hacíamos *inferencia*), aquí **afinaremos** (*fine-tuning*) un modelo preentrenado para nuestra propia tarea. El flujo completo es:

1. **Cargar** un dataset etiquetado (IMDb: reseñas de cine positivas/negativas).
2. **Tokenizar** el texto con el tokenizer del modelo base (DistilBERT).
3. **Preparar** los lotes con *padding dinámico*.
4. **Cargar** el modelo con una cabeza de clasificación y **entrenarlo**.
5. **Evaluar** e **inferir** sobre texto nuevo.

Empezamos instalando las tres librerías del ecosistema que necesitaremos: `transformers` (modelos), `datasets` (datos) y `evaluate` (métricas).

In [ ]:
# Instalación de las librerías del ecosistema HuggingFace (ejecutar una sola vez)
!pip install transformers datasets evaluate

## 2. Autenticación en el Hub

Para **publicar** el modelo entrenado en el Hub de HuggingFace (y recuperarlo luego) necesitamos iniciar sesión con un token de acceso. Este paso es opcional si solo quieres entrenar localmente.

In [ ]:
# Iniciar sesión en el Hub de HuggingFace desde el notebook
from huggingface_hub import notebook_login

notebook_login()

## 3. El dataset IMDb

[IMDb](https://huggingface.co/datasets/imdb) es un conjunto clásico para **clasificación binaria de sentimiento**: 25 000 reseñas de cine para entrenamiento y 25 000 para prueba, cada una etiquetada como positiva (`1`) o negativa (`0`). La función `load_dataset` lo descarga y lo deja listo como un objeto `DatasetDict`.

In [ ]:
# Cargar el dataset IMDb desde HuggingFace Datasets
from datasets import load_dataset

imdb = load_dataset("imdb")

## 4. Tokenización con `AutoTokenizer`

Un modelo no recibe texto crudo sino **tokens** (índices enteros). El **tokenizer** divide el texto en unidades de subpalabra (*subword*) y las mapea a IDs, usando el mismo vocabulario con el que se entrenó el modelo. Aquí usamos **DistilBERT**, una versión "destilada" (más pequeña y rápida, ~40% menos parámetros) de BERT que conserva la mayor parte de su desempeño.

In [ ]:
# Cargar el tokenizer preentrenado de DistilBERT
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

### Aplicar el tokenizer a todo el dataset

Definimos una función que tokeniza el campo `text`. El parámetro `truncation=True` recorta las reseñas que superan el largo máximo que admite el modelo. Luego la aplicamos a todo el dataset con `.map(..., batched=True)`, que procesa los ejemplos en lotes para mayor velocidad.

In [ ]:
# Funcion que tokeniza el texto, truncando las secuencias muy largas
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [ ]:
# Aplicar la tokenizacion a todo el dataset (por lotes)
tokenized_imdb = imdb.map(preprocess_function, batched=True)

## 5. Padding dinámico con *Data Collator*

Las secuencias tienen largos distintos, pero un lote (*batch*) debe ser rectangular. En vez de rellenar (*padding*) todas las reseñas al largo máximo global —lo que desperdicia cómputo—, el `DataCollatorWithPadding` rellena **cada lote** al largo de su ejemplo más largo (*padding dinámico*).

In [ ]:
# Data collator con padding dinamico por lote (tensores de TensorFlow)
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

## 6. Métrica de evaluación

Usaremos la **exactitud** (*accuracy*): la fracción de reseñas clasificadas correctamente. La librería `evaluate` la provee lista para usar; `compute_metrics` convierte los *logits* del modelo en etiquetas (`argmax`) antes de compararlas con las verdaderas.

In [ ]:
# Cargar la metrica de exactitud (accuracy)
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
# Convertir logits a etiquetas (argmax) y calcular la exactitud
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

## 7. Mapeo de etiquetas

El modelo trabaja con índices (`0`, `1`); estos diccionarios traducen entre el índice y su nombre legible. Se guardan en la configuración del modelo para que las salidas sean interpretables (`POSITIVE` / `NEGATIVE`).

In [ ]:
# Diccionarios de traduccion entre indice numerico y etiqueta legible
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

## 8. Optimizador y *schedule* de learning rate

`create_optimizer` construye un optimizador AdamW junto con un *schedule* que **decae** linealmente la tasa de aprendizaje a lo largo del entrenamiento. Calculamos el número total de pasos como (lotes por época) × (número de épocas).

In [ ]:
# Crear el optimizador con decaimiento lineal del learning rate
from transformers import create_optimizer
import tensorflow as tf

batch_size = 16
num_epochs = 5
batches_per_epoch = len(tokenized_imdb["train"]) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)
optimizer, schedule = create_optimizer(init_lr=2e-5, num_warmup_steps=0, num_train_steps=total_train_steps)

## 9. Carga del modelo con cabeza de clasificación

`TFAutoModelForSequenceClassification` toma el **encoder** preentrenado de DistilBERT y le añade una **cabeza de clasificación** (una capa densa) con `num_labels=2` salidas. Los pesos del encoder vienen de BERT; los de la cabeza se inicializan al azar y se aprenderán durante el fine-tuning.

In [ ]:
# Cargar DistilBERT con una cabeza de clasificacion de 2 clases
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
)

## 10. Conversión a `tf.data.Dataset`

`prepare_tf_dataset` transforma el dataset tokenizado en el formato eficiente `tf.data` que Keras consume durante el entrenamiento, aplicando el *data collator* a cada lote.

In [ ]:
# Preparar los conjuntos de entrenamiento y validacion como tf.data.Dataset
tf_train_set = model.prepare_tf_dataset(
    tokenized_imdb["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator,
)

tf_validation_set = model.prepare_tf_dataset(
    tokenized_imdb["test"],
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator,
)

## 11. Compilación del modelo

Compilamos con el optimizador definido. Nota que **no** pasamos un argumento `loss`: los modelos de `transformers` calculan **internamente** la función de pérdida apropiada para la tarea.

In [ ]:
# Compilar el modelo (la perdida se calcula internamente, sin argumento loss)
import tensorflow as tf

model.compile(optimizer=optimizer)  # No loss argument!

## 12. Callbacks de entrenamiento

Configuramos dos *callbacks*: `KerasMetricCallback` calcula la exactitud sobre el conjunto de validación al final de cada época, y `PushToHubCallback` sube automáticamente el modelo al Hub.

In [ ]:
# Callback que evalua la metrica (accuracy) en validacion tras cada epoca
from transformers.keras_callbacks import KerasMetricCallback

metric_callback = KerasMetricCallback(metric_fn=compute_metrics, eval_dataset=tf_validation_set)

In [ ]:
# Callback que publica el modelo en el Hub de HuggingFace durante el entrenamiento
from transformers.keras_callbacks import PushToHubCallback

push_to_hub_callback = PushToHubCallback(
    output_dir="text_classifier",
    tokenizer=tokenizer,
)

In [ ]:
# Agrupar los callbacks en una lista para pasarlos a fit()
callbacks = [metric_callback, push_to_hub_callback]

## 13. Fine-tuning

Entrenamos el modelo. Cada época recorre las 25 000 reseñas de entrenamiento; gracias al *transfer learning* bastan pocas épocas para alcanzar buena exactitud, ya que el encoder ya "entiende" el lenguaje.

In [ ]:
# Entrenar (fine-tuning) el modelo durante 3 epocas
#model.fit(x=tf_train_set, validation_data=tf_validation_set, epochs=3, callbacks=callbacks)
model.fit(x=tf_train_set, validation_data=tf_validation_set, epochs=3)

## 14. Inferencia con el modelo afinado

Con el modelo ya entrenado (y publicado en el Hub) lo aplicamos a una reseña nueva. Hay dos formas equivalentes:

- **Con `pipeline`**: la vía de alto nivel, en una línea.
- **Manual**: tokenizar → obtener *logits* → tomar el `argmax`. Útil para entender qué hace el `pipeline` por dentro.

In [ ]:
# Resena de ejemplo para probar el clasificador
text = "This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three."

In [ ]:
# Via rapida: clasificar el texto con un pipeline sobre el modelo publicado
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="polivares/text_classifier")
classifier(text)

In [ ]:
# Via manual (1/3): tokenizar el texto de entrada
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("polivares/text_classifier")
inputs = tokenizer(text, return_tensors="tf")

In [ ]:
# Via manual (2/3): obtener los logits del modelo
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained("polivares/text_classifier")
logits = model(**inputs).logits

In [ ]:
# Via manual (3/3): elegir la clase de mayor logit y traducirla a etiqueta
predicted_class_id = int(tf.math.argmax(logits, axis=-1)[0])
model.config.id2label[predicted_class_id]

## 📌 Resumen

| Etapa | Herramienta | Rol |
|-------|-------------|-----|
| Datos | `datasets.load_dataset` | Cargar IMDb |
| Tokenización | `AutoTokenizer` (DistilBERT) | Texto → tokens |
| Lotes | `DataCollatorWithPadding` | Padding dinámico |
| Modelo | `TFAutoModelForSequenceClassification` | Encoder + cabeza de clasificación |
| Entrenamiento | `model.fit` + `create_optimizer` | Fine-tuning |
| Métrica | `evaluate` (accuracy) | Evaluación |
| Inferencia | `pipeline` / tokenizer + modelo | Predicción sobre texto nuevo |

**Idea central:** el *fine-tuning* reutiliza un encoder preentrenado (que ya "entiende" el lenguaje) y solo aprende la cabeza de clasificación para la tarea específica, logrando buen desempeño con relativamente pocos datos y épocas.

## 📚 Referencias

- **HuggingFace.** *Text classification* (tutorial oficial). https://huggingface.co/docs/transformers/tasks/sequence_classification
- **Sanh, V. et al. (2019).** *DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter*. https://arxiv.org/abs/1910.01108
- **Devlin, J. et al. (2019).** *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL. https://arxiv.org/abs/1810.04805
- **Maas, A. et al. (2011).** *Learning Word Vectors for Sentiment Analysis* (dataset IMDb). ACL. https://ai.stanford.edu/~amaas/data/sentiment/
- **HuggingFace.** Documentación de `datasets` y `evaluate`. https://huggingface.co/docs